In [1]:

# 1. Setup & Load Data

import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split

# Define Paths (Dynamic)
# We go 'up' one level (..) to get out of 'notebooks' folder
BASE_DIR = os.path.dirname(os.getcwd()) 
DATA_PATH = os.path.join(BASE_DIR, 'data_generation', 'raw_data.csv')
ARTIFACTS_DIR = os.path.join(BASE_DIR, 'models')

# Load Data
print(f" Loading data from: {DATA_PATH}")
df = pd.read_csv(DATA_PATH)

print(f"   Shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")
display(df.head(3))

 Loading data from: c:\Users\rudra\OneDrive\Documents\Hackathon 3 Og\data_generation\raw_data.csv
   Shape: (1000, 11)
   Columns: ['Application_ID', 'Applicant_Name', 'Family_Income', 'Caste_Category', 'Domicile_Maharashtra', 'Mh_CET_Percentile', 'JEE_Percentile', 'University_Test_Score', '12th_Percentage', 'Extracurricular_Score', 'Scholarship_Priority_Score']


,Application_ID,Applicant_Name,Family_Income,Caste_Category,Domicile_Maharashtra,Mh_CET_Percentile,JEE_Percentile,University_Test_Score,12th_Percentage,Extracurricular_Score,Scholarship_Priority_Score
0,APP-2025-0001,Sai Patil,596019,General,1,78.83,48.59,100.0,71.3,2,54.5
1,APP-2025-0002,Isha Kulkarni,407192,General,1,44.44,49.07,54.3,77.1,5,60.3
2,APP-2025-0003,Diya Joshi,652530,OBC,1,62.49,65.92,58.8,65.8,1,59.7


In [2]:

# 2. Feature Selection

# A. Define Target
target_col = 'Scholarship_Priority_Score'

# B. Drop Identifiers (Name, ID) & Target from Input (X)
# We strictly select ONLY the features we want the model to learn
features_to_use = [
    'Family_Income', 
    'Caste_Category', 
    'Domicile_Maharashtra', 
    'Mh_CET_Percentile', 
    'JEE_Percentile', 
    'University_Test_Score', 
    '12th_Percentage', 
    'Extracurricular_Score'
]

print(f"Selected {len(features_to_use)} Features for Training:")
print(features_to_use)


Selected 8 Features for Training:
['Family_Income', 'Caste_Category', 'Domicile_Maharashtra', 'Mh_CET_Percentile', 'JEE_Percentile', 'University_Test_Score', '12th_Percentage', 'Extracurricular_Score']


In [3]:
X = df[features_to_use].copy()
y = df[target_col].copy()

# C. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\n Data Split:")
print(f"   Training Data: {X_train.shape}")
print(f"   Test Data:     {X_test.shape}")


 Data Split:
   Training Data: (800, 8)
   Test Data:     (200, 8)


In [4]:

# 3. Feature Engineering Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


In [5]:
# 1. Define Groups
numerical_features = [
    'Family_Income', 'Mh_CET_Percentile', 'JEE_Percentile', 
    'University_Test_Score', '12th_Percentage', 'Extracurricular_Score'
]

categorical_features = ['Caste_Category']

In [6]:
# 2. Build Transformer
# We do NOT include 'remainder=passthrough' blindly anymore.
# We explicitly handle Domicile (which is already 0/1) by letting it pass.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough' # This will keep 'Domicile_Maharashtra' as is
)

In [7]:
# 3. Fit on TRAIN data
print("  Fitting Preprocessor...")
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"   Original Shape:  {X_train.shape}")
print(f"   Processed Shape: {X_train_processed.shape} (Expanded due to One-Hot Encoding)")

  Fitting Preprocessor...
   Original Shape:  (800, 8)
   Processed Shape: (800, 11) (Expanded due to One-Hot Encoding)


In [8]:

# 4. Save Artifacts

import os

# Create models folder if not exists
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# A. Save the Preprocessor Object
joblib.dump(preprocessor, os.path.join(ARTIFACTS_DIR, 'preprocessor.pkl'))
print(f"Saved Preprocessor: {ARTIFACTS_DIR}/preprocessor.pkl")

# B. Save Processed Data (Ready for the Training Notebook)
# We save these as .npy files to make the next step faster
np.save(os.path.join(ARTIFACTS_DIR, 'X_train.npy'), X_train_processed)
np.save(os.path.join(ARTIFACTS_DIR, 'X_test.npy'), X_test_processed)
np.save(os.path.join(ARTIFACTS_DIR, 'y_train.npy'), y_train.to_numpy())
np.save(os.path.join(ARTIFACTS_DIR, 'y_test.npy'), y_test.to_numpy())

print(" Data Processing Complete.")

Saved Preprocessor: c:\Users\rudra\OneDrive\Documents\Hackathon 3 Og\models/preprocessor.pkl
 Data Processing Complete.
